# Fase 8 — Evaluación de LLMs

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USUARIO/REPO/blob/main/notebooks/08_evaluacion.ipynb)

> Medir un LLM con rigor: perplexity, BLEU/ROUGE/BERTScore (con ejemplos adversariales),
> alucinaciones, toxicidad, sesgo, robustez y un **benchmark propio multiple-choice**
> de los 21 dominios del lab.

## Objetivos de aprendizaje
- Derivar e implementar **perplexity** con sliding window y medirla por dominio.
- Descomponer **BLEU** n-grama a n-grama con tabla de cálculo manual.
- Visualizar la **matriz DP de LCS** usada por ROUGE-L.
- Comparar **BERTScore** con un heatmap de similitud token × token.
- Diseñar ~10 pares **adversariales** que revelan los límites de cada métrica.
- Detectar **alucinaciones** por self-consistency.
- Medir **sesgo** (WEAT) con estadístico de effect size completo.
- Construir un **harness de robustez** con múltiples tipos de perturbación.
- Construir un **benchmark propio** de ~50 preguntas con radar plotly y Wilson CI.
- Ensamblar un **dashboard 2×3** con todas las métricas.

## Prerrequisitos
Fases 1 – 7. Tiempo estimado: ~45 min SMALL · ~70 min LARGE.

## Índice
1. Por qué evaluar LLMs es difícil
2. Perplexity (sliding window, por dominio, comparabilidad entre tokenizadores)
3. BLEU desde cero (tabla de n-gramas, brevity penalty)
4. ROUGE-L (programación dinámica, heatmap de la matriz LCS)
5. BERTScore (heatmap token × token)
6. Ejemplos adversariales (~10 pares, tabla comparativa 4 métricas)
7. Alucinaciones (self-consistency)
8. Toxicidad
9. Sesgo — WEAT (effect size completo, barras divergentes)
10. Robustez (harness multi-perturbación, gráfico de degradación)
11. Benchmarks estándar
12. Benchmark propio (50 preguntas, radar plotly, heatmap modelo×dominio)
13. Comparación entre checkpoints
14. Dashboard final (figura 2×3)
15. Resumen, ejercicios y referencias

In [ ]:
# === Setup ===
import sys, subprocess, math, json, random, re, itertools
from collections import Counter, defaultdict

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# torch
try:
    import torch
except ImportError:
    _pip("torch"); import torch

# numpy / matplotlib / seaborn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    _pip("seaborn"); import seaborn as sns; sns.set_theme(style="whitegrid")

# plotly
try:
    import plotly.graph_objects as go
    import plotly.express as px
    PLOTLY = True
except ImportError:
    _pip("plotly"); import plotly.graph_objects as go
    import plotly.express as px; PLOTLY = True

# detect Colab
try:
    import google.colab; IN_COLAB = True
except ImportError:
    IN_COLAB = False

# device
DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()
          else "cpu")

# seeds
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "figure.dpi": 100})

print(f"Setup OK | device={DEVICE} | Colab={IN_COLAB}")

In [ ]:
# === Perfil SMALL / LARGE ===
# Cambia PROFILE = "small" | "large" | "auto"
PROFILE = "auto"

if PROFILE == "auto":
    if DEVICE == "cuda":
        gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        PROFILE = "large" if gb >= 12 else "small"
    else:
        PROFILE = "small"

CONFIG = {
    "small": {
        "model_name": "sshleifer/tiny-gpt2",   # ~117 K parámetros
        "stride": 64,
        "max_tokens_ppl": 512,
        "bench_reps": 2,                        # repeticiones del benchmark
    },
    "large": {
        "model_name": "gpt2",                   # 117 M parámetros
        "stride": 256,
        "max_tokens_ppl": 2048,
        "bench_reps": 5,
    },
}[PROFILE]

print(f"Perfil activo: {PROFILE.upper()}")
print(f"  modelo HF    : {CONFIG['model_name']}")
print(f"  stride       : {CONFIG['stride']}")

## 1. Por qué evaluar LLMs es difícil

### El problema central
A diferencia de clasificación (una sola etiqueta correcta), en generación
**muchas respuestas son igualmente válidas**. Las métricas automáticas intentan
aproximar el juicio humano, pero cada una captura solo una dimensión.

### Taxonomía de la evaluación

```mermaid
flowchart TB
    E[Evaluación de LLMs] --> I[Intrínseca\nperplexity]
    E --> X[Extrínseca / downstream]
    X --> A[Automática\nBLEU · ROUGE · BERTScore\nbenchmarks multiple-choice]
    X --> H[Humana\nA/B · raters]
    X --> J[LLM-as-judge\nGPT-4 · Claude]
```

### Problemas clave
| Problema | Descripción |
|----------|-------------|
| **Contaminación** | Los benchmarks filtran al corpus de entrenamiento → resultados inflados |
| **Ley de Goodhart** | "Cuando una métrica se vuelve objetivo, deja de ser buena métrica" |
| **Sensibilidad al prompt** | Cambiar el formato del prompt puede mover ±10 pp de accuracy |
| **Comparación entre tokenizadores** | La PPL **no** es comparable si los vocabularios difieren |
| **Evaluación costosa** | La evaluación humana escala mal; LLM-as-judge hereda los sesgos del juez |

> **Principio del lab:** implementar primero la métrica a mano → luego verificar con la librería → recién
> entonces interpretar los números.

## 2. Perplexity

### Definición

Dado un modelo de lenguaje $p$ y una secuencia $x_1,\ldots,x_N$:

$$\text{PPL}(x) = \exp\!\left(-\frac{1}{N}\sum_{t=1}^{N}\log p(x_t \mid x_{<t})\right) = \exp\bigl(\mathcal{H}\bigr)$$

donde $\mathcal{H}$ es la entropía cruzada promedio. Es el **branching factor efectivo**:
cuántos tokens equiprobables considera el modelo en cada paso. Menor = mejor.

### Sliding window para textos largos

Si el texto supera el contexto máximo $L$, se divide en ventanas solapadas:

$$\text{PPL}_{\text{SW}} = \exp\!\left(-\frac{1}{N}\sum_{i} \text{NLL}(\text{ventana}_i)\right)$$

Con **stride $s < L$**: cada token se predice en el contexto más largo disponible.
Cuando $s = 1$ es lo óptimo teórico; cuando $s = L$ (sin overlap) infraestima el
contexto y sube la PPL. La curva PPL vs stride muestra el trade-off costo/precisión.

### ¿Por qué la PPL no es comparable entre tokenizadores?

$$\text{PPL} = \exp\!\left(-\frac{1}{N_{\text{tok}}}\sum \log p\right)$$

Si el tokenizador A produce 120 tokens para un texto y el B produce 80,
la normalización $1/N$ es diferente: estamos promediando sobre cantidades
distintas. Un tokenizador más agresivo (subword más largo) produce PPL
artificialmente más baja.

In [ ]:
# Textos inline por dominio (2-3 párrafos por dominio, 21 dominios)
# Generados sintéticamente para no depender de descargas.

DOMAIN_TEXTS = {
    "machine_learning": (
        "El aprendizaje automático permite a las computadoras aprender patrones a partir "
        "de datos sin ser programadas explícitamente. Los algoritmos supervisados requieren "
        "etiquetas durante el entrenamiento. "
        "La regularización L2 penaliza los pesos grandes para reducir el sobreajuste. "
        "El validación cruzada estima el error de generalización dividiendo el conjunto de datos."
    ),
    "deep_learning": (
        "Las redes neuronales profundas aprenden representaciones jerárquicas. "
        "La retropropagación calcula los gradientes mediante la regla de la cadena. "
        "El dropout desactiva neuronas aleatoriamente para regularización. "
        "Las capas de normalización por lotes estabilizan el entrenamiento profundo."
    ),
    "inteligencia_artificial": (
        "La inteligencia artificial busca replicar capacidades cognitivas en máquinas. "
        "Los sistemas de búsqueda A* encuentran el camino óptimo en grafos. "
        "Los modelos de lenguaje grande generan texto coherente a partir de un prompt. "
        "El razonamiento simbólico combina lógica formal con representación del conocimiento."
    ),
    "ingenieria_electrica": (
        "La ley de Ohm establece que la corriente es proporcional al voltaje e inversamente "
        "proporcional a la resistencia. Los circuitos RLC presentan resonancia en frecuencia. "
        "El análisis de nodos aplica la ley de Kirchhoff de corrientes para resolver circuitos. "
        "Las máquinas síncronas convierten energía mecánica en energía eléctrica alterna."
    ),
    "sistemas_potencia": (
        "El flujo de carga calcula los voltajes nodales del sistema de potencia en estado estable. "
        "El método de Newton-Raphson converge cuadráticamente en problemas de flujo de carga. "
        "La estabilidad transitoria analiza la respuesta del sistema ante perturbaciones severas. "
        "Las líneas de transmisión de alta tensión minimizan las pérdidas por efecto Joule."
    ),
    "ciencia_datos": (
        "El análisis exploratorio de datos revela distribuciones, outliers y correlaciones. "
        "El pipeline de datos incluye ingestión, limpieza, transformación y carga. "
        "La visualización efectiva comunica patrones complejos de forma intuitiva. "
        "Los conjuntos de datos desequilibrados requieren técnicas especiales de muestreo."
    ),
    "matematicas": (
        "La descomposición en valores singulares factoriza cualquier matriz como A = UΣVᵀ. "
        "El teorema de Bayes actualiza probabilidades a priori con evidencia observada. "
        "La transformada de Fourier descompone señales en sus componentes de frecuencia. "
        "Los espacios vectoriales son conjuntos con operaciones de suma y multiplicación escalar."
    ),
    "estadistica": (
        "El teorema central del límite establece que la media muestral se distribuye normalmente. "
        "El intervalo de confianza cuantifica la incertidumbre de un estimador puntual. "
        "El test de hipótesis decide si rechazar la hipótesis nula con un nivel de significancia. "
        "La regresión lineal modela la relación entre variables dependientes e independientes."
    ),
    "optimizacion": (
        "El descenso de gradiente actualiza los parámetros en dirección del gradiente negativo. "
        "Adam combina momentum y adaptación de la tasa de aprendizaje por parámetro. "
        "Los métodos de punto interior resuelven problemas de programación convexa. "
        "La programación lineal optimiza una función objetivo sobre restricciones lineales."
    ),
    "python": (
        "Las comprensiones de lista generan listas de forma concisa y eficiente en Python. "
        "Los decoradores modifican el comportamiento de funciones sin alterar su código. "
        "Los generadores producen valores de forma perezosa para ahorrar memoria. "
        "El módulo asyncio permite programación asíncrona con corutinas en Python."
    ),
    "programacion": (
        "Los patrones de diseño son soluciones reutilizables a problemas de software comunes. "
        "La complejidad temporal Big-O describe cómo escala un algoritmo con la entrada. "
        "La recursión divide un problema en subproblemas más pequeños del mismo tipo. "
        "Las estructuras de datos como árboles y grafos organizan información jerárquica."
    ),
    "postgresql": (
        "El índice B-tree de PostgreSQL acelera las consultas de rango y de igualdad. "
        "VACUUM recupera el espacio de tuplas muertas después de operaciones DELETE. "
        "Las transacciones garantizan las propiedades ACID en operaciones concurrentes. "
        "El planificador de consultas elige el plan de ejecución con menor coste estimado."
    ),
    "mongodb": (
        "MongoDB almacena documentos BSON en colecciones sin esquema fijo predefinido. "
        "Los índices compuestos aceleran consultas que filtran por múltiples campos. "
        "El pipeline de agregación permite transformaciones y agrupaciones complejas. "
        "La replicación mediante replica sets proporciona alta disponibilidad automática."
    ),
    "aws": (
        "AWS Lambda ejecuta código sin aprovisionar ni gestionar servidores subyacentes. "
        "Amazon S3 ofrece almacenamiento de objetos con durabilidad de once nueves. "
        "El auto scaling ajusta la capacidad de cómputo según la demanda en tiempo real. "
        "CloudFormation define la infraestructura como código mediante plantillas declarativas."
    ),
    "serverless": (
        "El framework Serverless simplifica el despliegue de funciones Lambda con YAML. "
        "Los eventos de API Gateway desencadenan funciones Lambda en respuesta a HTTP. "
        "El cold start introduce latencia adicional cuando se inicializa una función fría. "
        "Las capas Lambda permiten compartir código y dependencias entre funciones."
    ),
    "iac": (
        "Terraform gestiona infraestructura en múltiples proveedores mediante HCL declarativo. "
        "El estado de Terraform registra los recursos creados para detectar cambios. "
        "Los módulos reutilizan configuraciones de infraestructura entre proyectos distintos. "
        "Ansible automatiza la configuración de servidores con playbooks en YAML."
    ),
    "apis": (
        "REST usa verbos HTTP para operaciones CRUD sobre recursos identificados por URI. "
        "GraphQL permite al cliente especificar exactamente los campos que necesita. "
        "La autenticación OAuth delega el acceso sin compartir credenciales directamente. "
        "Las APIs deben versionar sus endpoints para mantener compatibilidad con clientes."
    ),
    "arquitectura_software": (
        "Los microservicios descomponen una aplicación en servicios independientes y desplegables. "
        "El patrón CQRS separa las operaciones de lectura y escritura para mayor escalabilidad. "
        "Event sourcing almacena el estado como secuencia de eventos inmutables. "
        "El patrón circuit breaker previene cascadas de fallos en sistemas distribuidos."
    ),
    "devops": (
        "La integración continua valida automáticamente cada commit mediante tests y linting. "
        "Los contenedores Docker garantizan portabilidad entre entornos de desarrollo y producción. "
        "Kubernetes orquesta contenedores con escalado, reinicio y balanceo de carga. "
        "El monitoreo con métricas y alertas detecta degradaciones antes de afectar usuarios."
    ),
    "data_engineering": (
        "Los pipelines ETL extraen, transforman y cargan datos entre sistemas heterogéneos. "
        "Apache Spark procesa grandes volúmenes de datos en paralelo sobre un clúster. "
        "El modelado dimensional organiza datos en tablas de hechos y dimensiones. "
        "La calidad de datos mide completitud, consistencia y precisión del conjunto."
    ),
    "sistemas_distribuidos": (
        "El teorema CAP establece que un sistema distribuido no puede garantizar simultáneamente "
        "consistencia, disponibilidad y tolerancia a particiones de red. "
        "El consenso de Raft elige un líder y replica el log entre nodos del clúster. "
        "La consistencia eventual permite divergencia temporal a cambio de mayor disponibilidad."
    ),
}

print(f"Dominios cargados: {len(DOMAIN_TEXTS)}")
for dom, txt in DOMAIN_TEXTS.items():
    print(f"  {dom:30s}: {len(txt):4d} chars")

In [ ]:
# Carga modelo HF para calcular PPL (try/except con fallback unigram)

def hf_ppl_domain(model, tokenizer, text, stride, max_tokens):
    """Perplexity con sliding window sobre 'text'."""
    ids = tokenizer(text, return_tensors="pt").input_ids.to(DEVICE)
    ids = ids[:, :max_tokens]
    seq_len = ids.size(1)
    max_ctx = getattr(model.config, "n_positions",
               getattr(model.config, "max_position_embeddings", 128))
    stride = min(stride, max_ctx)
    nlls = []
    prev_end = 0
    for begin in range(0, seq_len, stride):
        end = min(begin + max_ctx, seq_len)
        trg_len = end - prev_end
        chunk = ids[:, begin:end]
        target = chunk.clone()
        target[:, :-trg_len] = -100
        with torch.no_grad():
            loss = model(chunk, labels=target).loss
        nlls.append(loss.item() * trg_len)
        prev_end = end
        if end == seq_len:
            break
    return math.exp(sum(nlls) / seq_len)


def compute_ppl_hf():
    _pip("transformers", "accelerate")
    from transformers import AutoModelForCausalLM, AutoTokenizer
    name = CONFIG["model_name"]
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForCausalLM.from_pretrained(name).to(DEVICE).eval()
    results = {}
    for dom, txt in DOMAIN_TEXTS.items():
        results[dom] = hf_ppl_domain(
            mdl, tok, txt,
            stride=CONFIG["stride"],
            max_tokens=CONFIG["max_tokens_ppl"]
        )
    return results, name


def compute_ppl_unigram():
    """Fallback: modelo unigram Laplace sobre cada dominio."""
    all_words = [w for t in DOMAIN_TEXTS.values() for w in t.lower().split()]
    cnt = Counter(all_words); V = len(cnt); N = sum(cnt.values())
    def _ppl(text):
        words = text.lower().split()
        lp = sum(math.log((cnt[w] + 1) / (N + V)) for w in words)
        return math.exp(-lp / max(len(words), 1))
    return {dom: _ppl(txt) for dom, txt in DOMAIN_TEXTS.items()}, "unigram-Laplace"


try:
    PPL_RESULTS, PPL_SRC = compute_ppl_hf()
    print(f"PPL calculada con {PPL_SRC}")
except Exception as exc:
    print(f"HF no disponible ({type(exc).__name__}), usando unigram Laplace.")
    PPL_RESULTS, PPL_SRC = compute_ppl_unigram()

for dom, ppl in sorted(PPL_RESULTS.items(), key=lambda x: x[1]):
    print(f"  {dom:35s}: {ppl:8.2f}")

In [ ]:
# Barras ordenadas de PPL por dominio
sorted_items = sorted(PPL_RESULTS.items(), key=lambda x: x[1])
doms_sorted = [d for d, _ in sorted_items]
ppl_sorted  = [v for _, v in sorted_items]

# Colores por peso del dominio (corpus.yaml)
DOMAIN_WEIGHTS = {
    "machine_learning":1.5,"deep_learning":1.5,"inteligencia_artificial":1.5,
    "ingenieria_electrica":1.5,"sistemas_potencia":1.5,
    "ciencia_datos":1.2,"matematicas":1.2,"estadistica":1.2,
    "optimizacion":1.2,"python":1.2,
    "programacion":1.0,"postgresql":1.0,"mongodb":1.0,"aws":1.0,
    "serverless":1.0,"iac":1.0,"apis":1.0,"arquitectura_software":1.0,
    "devops":1.0,"data_engineering":1.0,"sistemas_distribuidos":1.0,
}
palette = {1.5:"#1a6fa8", 1.2:"#3aaa55", 1.0:"#c47a1e"}
colors = [palette[DOMAIN_WEIGHTS.get(d, 1.0)] for d in doms_sorted]

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.barh(doms_sorted, ppl_sorted, color=colors)
ax.set_xlabel("Perplexity")
ax.set_title(f"Perplexity por dominio — {PPL_SRC}\n"
             "Azul=peso 1.5 · Verde=peso 1.2 · Naranja=peso 1.0")
for bar, val in zip(bars, ppl_sorted):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}", va="center", fontsize=8)
plt.tight_layout(); plt.show()
print("Nota: diferencias entre dominios reflejan familiaridad del modelo con el vocabulario,")
print("NO necesariamente dificultad intrínseca del dominio.")

### 2.1 Curva PPL vs stride

A mayor stride (menos overlap entre ventanas), cada token se predice con
menos contexto → la PPL sube. El trade-off es **velocidad vs precisión**:

- stride = 1 → óptimo pero $O(N^2)$ en tiempo
- stride = L (contexto completo) → más rápido pero PPL sobreestimada
- stride ≈ L/2 → punto de equilibrio habitual en la práctica

In [ ]:
# Curva PPL vs stride para un texto representativo
try:
    _pip("transformers")
    from transformers import AutoModelForCausalLM, AutoTokenizer
    _name = CONFIG["model_name"]
    _tok = AutoTokenizer.from_pretrained(_name)
    _mdl = AutoModelForCausalLM.from_pretrained(_name).to(DEVICE).eval()
    _max_ctx = getattr(_mdl.config, "n_positions",
                getattr(_mdl.config, "max_position_embeddings", 128))
    _text = " ".join(list(DOMAIN_TEXTS.values())[:3])  # concatenar 3 dominios

    strides = [max(1, _max_ctx // k) for k in [8, 4, 2, 1]]
    strides = sorted(set(strides))
    ppl_vs_stride = []
    for s in strides:
        p = hf_ppl_domain(_mdl, _tok, _text, stride=s,
                          max_tokens=CONFIG["max_tokens_ppl"])
        ppl_vs_stride.append(p)
        print(f"  stride={s:4d}  PPL={p:.2f}")

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(strides, ppl_vs_stride, "o-", color="#1a6fa8", lw=2)
    ax.set_xlabel("stride (tokens)"); ax.set_ylabel("Perplexity")
    ax.set_title("PPL vs stride — texto multi-dominio")
    plt.tight_layout(); plt.show()

except Exception as exc:
    # Fallback: curva simulada
    print(f"HF no disponible ({type(exc).__name__}), mostrando curva simulada.")
    strides = [8, 16, 32, 64, 128]
    ppl_sim = [12.3, 13.1, 14.8, 17.2, 21.5]
    plt.figure(figsize=(7, 4))
    plt.plot(strides, ppl_sim, "o--", color="#1a6fa8")
    plt.xlabel("stride"); plt.ylabel("PPL (simulada)")
    plt.title("PPL vs stride (simulada — instala transformers para valor real)")
    plt.tight_layout(); plt.show()

### 2.2 Por qué la PPL no es comparable entre tokenizadores

$$\text{PPL}_A = \exp\!\left(-\frac{1}{N_A}\sum \log p_A\right), \quad
\text{PPL}_B = \exp\!\left(-\frac{1}{N_B}\sum \log p_B\right)$$

Si $N_A \ne N_B$ para el mismo texto, la normalización es distinta y la comparación es inválida.
La solución es reportar **bits-per-character (BPC)** o **bits-per-word**, que son invariantes
al tokenizador.

In [ ]:
# Demostración numérica: mismo texto, dos tokenizadores distintos
text_demo = ("El transformador de potencia convierte el voltaje de transmisión "
             "al nivel de distribución mediante inducción electromagnética.")

def mock_tokenize(text, vocab_size):
    """Tokenizador ficticio: divide en chunks de tamaño variable."""
    rng2 = np.random.default_rng(vocab_size)
    words = text.split()
    n_tokens = max(1, int(len(words) * (8000 / vocab_size) ** 0.3))
    return n_tokens

# Dos vocabularios distintos
for vsize, name_tok in [(8000, "SMALL-8k"), (32000, "LARGE-32k")]:
    n_tok = mock_tokenize(text_demo, vsize)
    # Supongamos la misma NLL total (mismo modelo, mismo texto)
    nll_total = 45.0   # número fijo para ilustrar
    ppl = math.exp(nll_total / n_tok)
    bpc = nll_total / (len(text_demo) * math.log(2))  # invariante
    print(f"Tokenizador {name_tok}: N_tokens={n_tok:3d}  PPL={ppl:8.2f}  BPC={bpc:.4f}")

print()
print("Conclusion: PPL difiere aunque NLL total sea identica.")
print("BPC (bits-per-character) es comparable entre tokenizadores.")

## 3. BLEU desde cero

### Definición

$$\text{BLEU} = \underbrace{\text{BP}}_{\text{brevity penalty}} \cdot \exp\!\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

donde $p_n$ es la precisión de n-gramas con **clipping**:

$$p_n = \frac{\sum_{\text{n-grama} \in \hat{y}} \min\!\bigl(\#_{\hat{y}}(\text{n-grama}),\, \#_y(\text{n-grama})\bigr)}{\sum_{\text{n-grama} \in \hat{y}} \#_{\hat{y}}(\text{n-grama})}$$

y la penalización de brevedad:

$$\text{BP} = \begin{cases} 1 & |\hat{y}| > |y| \\ e^{1-|y|/|\hat{y}|} & |\hat{y}| \le |y| \end{cases}$$

### Ejemplo manual (n=1..4)

Candidato: `"el transformador reduce el voltaje"`
Referencia: `"el transformador reduce el voltaje en la subestacion"`

La tabla siguiente desglosa cada n-grama.

In [ ]:
def ngrams(tokens, n):
    """Counter de todos los n-gramas en la lista de tokens."""
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))


def bleu_full(candidate: str, reference: str, max_n: int = 4, verbose: bool = True):
    """BLEU manual con tabla detallada n-grama a n-grama."""
    cand = candidate.lower().split()
    ref  = reference.lower().split()
    if not cand:
        return 0.0

    rows = []
    precs = []
    for n in range(1, max_n + 1):
        cg = ngrams(cand, n)
        rg = ngrams(ref, n)
        if not cg:
            precs.append(0.0)
            rows.append((n, "-", 0, 0, 0.0))
            continue
        total = sum(cg.values())
        clip  = sum(min(c, rg.get(g, 0)) for g, c in cg.items())
        pn    = clip / total if total > 0 else 0.0
        precs.append(pn)
        rows.append((n, str(dict(list(cg.items())[:3])) + "...",
                     total, clip, pn))

    geo = 0.0
    if min(precs) > 0:
        geo = math.exp(sum(math.log(p) for p in precs) / max_n)
    bp = 1.0 if len(cand) > len(ref) else math.exp(1 - len(ref)/max(len(cand), 1))
    score = bp * geo

    if verbose:
        print(f"Candidato : {candidate}")
        print(f"Referencia: {reference}")
        print(f"{'n':>3}  {'n-gramas (muestra)':35s}  {'total':>6}  {'clip':>5}  {'p_n':>6}")
        print("-" * 65)
        for n, sample, total, clip, pn in rows:
            print(f"{n:>3}  {sample:35s}  {total:6d}  {clip:5d}  {pn:6.3f}")
        print("-" * 65)
        print(f"BP={bp:.4f}  geo_mean={geo:.4f}  BLEU={score:.4f}")
    return score


BLEU_REF = "el transformador reduce el voltaje en la subestacion electrica"
BLEU_CAND = "el transformador reduce el voltaje"
score_bleu = bleu_full(BLEU_CAND, BLEU_REF)

## 4. ROUGE-L (Longest Common Subsequence)

ROUGE-L usa la **subsecuencia común más larga** (LCS) entre candidato y referencia:

$$P_{\text{LCS}} = \frac{\text{LCS}(c, r)}{|c|}, \quad
R_{\text{LCS}} = \frac{\text{LCS}(c, r)}{|r|}, \quad
F_{\text{LCS}} = \frac{(1+\beta^2)P_{\text{LCS}} R_{\text{LCS}}}{R_{\text{LCS}} + \beta^2 P_{\text{LCS}}}$$

Con $\beta = 1$ obtenemos la F1 estándar.

La LCS se calcula por programación dinámica en $O(|c| \cdot |r|)$ tiempo y espacio.
La siguiente celda construye la matriz DP y la visualiza como heatmap.

In [ ]:
def lcs_dp(a, b):
    """Matriz DP de LCS. dp[i][j] = LCS de a[:i] y b[:j]."""
    m, k = len(a), len(b)
    dp = np.zeros((m+1, k+1), dtype=int)
    for i in range(1, m+1):
        for j in range(1, k+1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp


def rouge_l(candidate: str, reference: str, beta: float = 1.0):
    c = candidate.lower().split()
    r = reference.lower().split()
    dp = lcs_dp(c, r)
    lcs = int(dp[-1, -1])
    P = lcs / max(len(c), 1)
    R = lcs / max(len(r), 1)
    denom = R + beta**2 * P
    F = (1 + beta**2) * P * R / denom if denom > 0 else 0.0
    return {"P": P, "R": R, "F": F, "LCS": lcs, "dp": dp, "c": c, "r": r}


cand_r = "el voltaje de la subestacion es reducido"
ref_r  = "el voltaje en la subestacion se reduce para distribucion"
res = rouge_l(cand_r, ref_r)
c_toks, r_toks, dp = res["c"], res["r"], res["dp"]

# Heatmap de la matriz DP
fig, ax = plt.subplots(figsize=(len(r_toks)*0.9 + 1, len(c_toks)*0.7 + 1))
sns.heatmap(dp[1:, 1:], annot=True, fmt="d", cmap="YlOrRd",
            xticklabels=r_toks, yticklabels=c_toks, ax=ax,
            cbar_kws={"label": "LCS parcial"})
ax.set_title(f"Matriz DP de LCS  (LCS={res['LCS']})\n"
             f"ROUGE-L: P={res['P']:.2f}  R={res['R']:.2f}  F={res['F']:.2f}")
ax.set_xlabel("referencia"); ax.set_ylabel("candidato")
plt.tight_layout(); plt.show()
print(f"ROUGE-L  P={res['P']:.3f}  R={res['R']:.3f}  F={res['F']:.3f}  LCS={res['LCS']}")

## 5. BERTScore

BLEU y ROUGE comparan tokens exactos — fallan con sinónimos y paráfrasis.
**BERTScore** usa embeddings contextuales para comparar significados:

$$P_{\text{BERT}} = \frac{1}{|\hat{y}|}\sum_{i} \max_{j} \cos(\mathbf{h}_i^{\hat{y}}, \mathbf{h}_j^y)$$

$$R_{\text{BERT}} = \frac{1}{|y|}\sum_{j} \max_{i} \cos(\mathbf{h}_i^{\hat{y}}, \mathbf{h}_j^y)$$

$$F_{\text{BERT}} = 2 \cdot \frac{P_{\text{BERT}} \cdot R_{\text{BERT}}}{P_{\text{BERT}} + R_{\text{BERT}}}$$

donde $\mathbf{h}_i$ son las representaciones del último layer de un encoder (BERT, RoBERTa).
El *greedy matching* asigna cada token al más similar del otro lado.

### Heatmap de similitud token × token

La siguiente celda usa embeddings de 4 dimensiones con sinónimos "plantados" a mano
para demostrar la intuición. En producción se usaría `bert-score` con un modelo real.

In [ ]:
# Embeddings de juguete con estructura semántica plantada
# dim 0: técnico/genérico, dim 1: acción/objeto, dim 2: positivo/negativo, dim 3: raro

VOCAB_EMB = {
    # función
    "reduce":    [0.8,  0.9,  0.5,  0.1],
    "baja":      [0.7,  0.9,  0.5,  0.2],   # sinónimo de reduce
    "aumenta":   [0.8,  0.9, -0.5,  0.1],   # antónimo
    # sujeto
    "transformador": [0.9, 0.3, 0.5, 0.1],
    "maquina":       [0.7, 0.3, 0.4, 0.3],  # hiperónimo
    "generador":     [0.8, 0.3, 0.5, 0.2],
    # objeto
    "voltaje":   [0.9, 0.2, 0.5, 0.0],
    "tension":   [0.9, 0.2, 0.5, 0.1],  # sinónimo de voltaje
    "corriente": [0.8, 0.2, 0.4, 0.2],
    # determinantes (poca carga semántica)
    "el":   [0.0, 0.0, 0.0, 0.0],
    "la":   [0.0, 0.0, 0.0, 0.0],
    "del":  [0.0, 0.0, 0.0, 0.1],
}

def get_emb(word):
    rng3 = np.random.default_rng(abs(hash(word)) % (2**32))
    base = np.array(VOCAB_EMB.get(word.lower(),
                    rng3.standard_normal(4).tolist()), dtype=float)
    norm = np.linalg.norm(base)
    return base / (norm + 1e-9)

def cos_sim(a, b):
    return float(np.dot(a, b))   # ya normalizados

cand_bs = "el transformador reduce el voltaje".split()
ref_bs  = "la maquina baja la tension del sistema".split()

# matriz de similitud |cand| x |ref|
S = np.array([[cos_sim(get_emb(c), get_emb(r)) for r in ref_bs] for c in cand_bs])

fig, ax = plt.subplots(figsize=(len(ref_bs)*1.1 + 0.5, len(cand_bs)*0.9 + 0.5))
sns.heatmap(S, annot=True, fmt=".2f", cmap="RdYlGn", vmin=-0.2, vmax=1.0,
            xticklabels=ref_bs, yticklabels=cand_bs, ax=ax,
            cbar_kws={"label": "cosine similarity"})
ax.set_title("BERTScore: similitud token × token\n"
             f"(embeddings 4-dim con estructura semántica plantada)")
ax.set_xlabel("Referencia"); ax.set_ylabel("Candidato")
plt.tight_layout(); plt.show()

# greedy matching
P_bs = S.max(axis=1).mean()
R_bs = S.max(axis=0).mean()
F_bs = 2*P_bs*R_bs/(P_bs+R_bs) if (P_bs+R_bs) > 0 else 0
print(f"BERTScore  P={P_bs:.3f}  R={R_bs:.3f}  F={F_bs:.3f}")
print("Observa: 'reduce'↔'baja' y 'voltaje'↔'tension' tienen alta similitud (sinónimos).")
print("BLEU daría 0 para este par. BERTScore lo captura correctamente.")

## 6. Ejemplos adversariales (~10 pares)

Los ejemplos adversariales revelan qué captura y qué ignora cada métrica:

| Tipo | Problema | Métrica afectada |
|------|----------|-----------------|
| **Negación** | Palabras iguales, significado opuesto | BLEU alto (¡falso!) |
| **Paráfrasis** | Mismo significado, palabras distintas | BLEU/ROUGE bajos (¡falso!) |
| **Sinónimos del dominio** | `voltaje` ↔ `tensión` | BLEU/ROUGE bajos |
| **Orden inverso** | Mismas palabras, sintaxis diferente | ROUGE-L baja |
| **Repetición** | Candidato repite palabras de ref | BLEU infla recall |
| **Truismo** | Respuesta siempre verdadera pero vacía | Todas las métricas fallan |

Los ~10 pares a continuación cubren estos casos con vocabulario eléctrico/ML.

In [ ]:
# ~10 pares adversariales con vocabulario eléctrico/ML
ADV_PAIRS = [
    # (candidato, referencia, tipo, descripcion)
    ("el interruptor esta abierto",
     "el interruptor esta cerrado",
     "negacion", "Antónimos eléctricos: mismo BLEU alto, significado opuesto"),

    ("el transformador eleva el voltaje",
     "el transformador reduce el voltaje",
     "negacion", "Verbo opuesto: eleva vs reduce"),

    ("la red neuronal aprende patrones en los datos",
     "el modelo de aprendizaje automatico extrae caracteristicas",
     "parafrasis", "Paráfrasis ML: mismo significado, palabras distintas"),

    ("el descenso de gradiente minimiza la funcion de perdida",
     "el optimizador actualiza los pesos para reducir el error",
     "parafrasis", "Paráfrasis optimización"),

    ("el voltaje en el nodo es alto",
     "la tension en el nodo es elevada",
     "sinonimos", "voltaje=tensión, alto=elevado (sinónimos dominio eléctrico)"),

    ("la resistencia del conductor aumenta con la temperatura",
     "la impedancia del cable crece al subir la temperatura",
     "sinonimos", "resistencia≈impedancia, conductor≈cable"),

    ("el modelo generaliza bien sobre datos no vistos",
     "bien datos no vistos el modelo sobre generaliza",
     "orden", "Palabras idénticas, orden barajado"),

    ("la corriente la corriente la corriente fluye",
     "la corriente fluye por el conductor",
     "repeticion", "Candidato repite para inflar n-gramas"),

    ("los sistemas electricos existen",
     "el flujo de carga calcula los voltajes nodales del sistema electrico",
     "truismo", "Candidato trivialmente verdadero pero sin información"),

    ("la transformada de fourier descompone la senal en frecuencias",
     "la transformada de fourier descompone la senal en frecuencias",
     "identico", "Candidato = Referencia (cota superior)"),
]

# Calcular BLEU, ROUGE-L y BERTScore-proxy para cada par
def bertscore_proxy(c, r):
    ct, rt = c.split(), r.split()
    if not ct or not rt:
        return 0.0
    S_mat = np.array([[cos_sim(get_emb(a), get_emb(b)) for b in rt] for a in ct])
    P_ = S_mat.max(axis=1).mean()
    R_ = S_mat.max(axis=0).mean()
    return 2*P_*R_/(P_+R_+1e-9)

results_adv = []
for cand, ref, tipo, desc in ADV_PAIRS:
    b = bleu_full(cand, ref, verbose=False)
    rl = rouge_l(cand, ref)["F"]
    bs = bertscore_proxy(cand, ref)
    results_adv.append({"tipo": tipo, "desc": desc[:45],
                        "BLEU": b, "ROUGE-L": rl, "BERTScore": bs})

# Tabla comparativa
print(f"{'Tipo':12s} {'Descripcion':45s}  {'BLEU':>6} {'ROUGE-L':>8} {'BERTScore':>10}")
print("-" * 90)
for r in results_adv:
    print(f"{r['tipo']:12s} {r['desc']:45s}  "
          f"{r['BLEU']:6.3f} {r['ROUGE-L']:8.3f} {r['BERTScore']:10.3f}")

In [ ]:
# Visualización de las 3 métricas sobre los pares adversariales
labels = [f"{r['tipo'][:6]}\n{r['desc'][:20]}..." for r in results_adv]
x = np.arange(len(results_adv))
width = 0.28

fig, ax = plt.subplots(figsize=(15, 5))
ax.bar(x - width, [r["BLEU"]     for r in results_adv], width, label="BLEU",      color="#1a6fa8")
ax.bar(x,         [r["ROUGE-L"]  for r in results_adv], width, label="ROUGE-L",   color="#3aaa55")
ax.bar(x + width, [r["BERTScore"]for r in results_adv], width, label="BERTScore", color="#c47a1e")
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=7)
ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
ax.set_title("Métricas sobre pares adversariales — revela sus fortalezas y debilidades")
ax.legend(); plt.tight_layout(); plt.show()

print()
print("Observaciones clave:")
print("  negacion  -> BLEU alto pese a significado opuesto")
print("  parafrasis-> BLEU/ROUGE bajos pese a mismo significado; BERTScore los captura")
print("  sinonimos -> BLEU=0, BERTScore > 0.5")
print("  repeticion-> BLEU infla la precisión con clipping parcial")

## 7. Alucinaciones

### Taxonomía
- **Intrínsecas**: contradicen la fuente que se le da al modelo.
- **Extrínsecas**: afirmaciones que no se pueden verificar con la fuente.
- **Factual**: contradicen el conocimiento del mundo.

### Detección por self-consistency

Se generan $N$ respuestas con temperatura $T>0$ y se mide el acuerdo:

$$\text{consistencia} = \frac{\max_a \text{count}(a)}{N}$$

Baja consistencia sugiere que el modelo "adivina" → posible alucinación.
Alta consistencia no garantiza veracidad (puede alucinarse de forma consistente).

In [ ]:
def consistency(answers):
    c = Counter(answers)
    return c.most_common(1)[0][1] / len(answers)

def entropy_answers(answers):
    """Entropía de la distribución de respuestas."""
    c = Counter(answers)
    total = len(answers)
    probs = [v/total for v in c.values()]
    return -sum(p*math.log(p+1e-9) for p in probs)

# Simulación: respuesta factual vs alucinada
qa_examples = [
    {
        "pregunta": "¿Cuál es el nivel de tensión típico de transmisión en Colombia?",
        "respuestas": ["500 kV", "500 kV", "500 kV", "500 kV", "220 kV"],
        "tipo": "factual",
    },
    {
        "pregunta": "¿Cuántos transformadores tiene la subestación Bacatá?",
        "respuestas": ["3", "5", "4", "2", "7"],
        "tipo": "alucinada",
    },
    {
        "pregunta": "¿Qué optimizador usa Adam internamente?",
        "respuestas": ["momentum+RMSProp", "momentum+RMSProp", "SGD", "momentum+RMSProp", "Adagrad"],
        "tipo": "mayoritaria",
    },
]

print(f"{'Tipo':12s} {'Consistencia':>13s} {'Entropía':>10s}  Pregunta")
print("-" * 75)
for ex in qa_examples:
    cons  = consistency(ex["respuestas"])
    entro = entropy_answers(ex["respuestas"])
    print(f"{ex['tipo']:12s} {cons:13.2f} {entro:10.3f}  {ex['pregunta'][:50]}")

# Gráfico
labels_hall = [e["tipo"] for e in qa_examples]
conss = [consistency(e["respuestas"]) for e in qa_examples]
cols  = ["green" if c >= 0.7 else "orange" if c >= 0.5 else "red" for c in conss]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels_hall, conss, color=cols)
ax.axhline(0.7, color="green",  ls="--", label="umbral confiable (0.7)")
ax.axhline(0.5, color="orange", ls="--", label="umbral dudoso (0.5)")
ax.set_ylabel("self-consistency"); ax.set_ylim(0, 1.1)
ax.set_title("Self-consistency como señal de alucinación")
ax.legend(); plt.tight_layout(); plt.show()

## 8. Toxicidad

### El problema
Los clasificadores de toxicidad son imperfectos:
- **Listas de palabras**: frágiles, no entienden contexto ni sarcasmo.
- **Modelos**: `detoxify` (Unitaryai), Perspective API (Google).
- **Trade-off**: mayor umbral → más safe pero más "over-refusals".

### Benchmark estándar
*RealToxicityPrompts* (Gehman et al., 2020): 100 K prompts con probabilidades de continuación tóxica.
Se reporta la tasa de generación tóxica y la *expected maximum toxicity* (EMT).

In [ ]:
# Demo: try/except detoxify, fallback lista de palabras
try:
    _pip("detoxify")
    from detoxify import Detoxify
    _dtox = Detoxify("original")
    def toxicity_score(text):
        return _dtox.predict(text)["toxicity"]
    TOX_SRC = "detoxify"
except Exception:
    # fallback: lista de palabras de riesgo
    _BANNED = {"idiota","estupido","maldito","inutil","basura"}
    def toxicity_score(text):
        words = set(text.lower().split())
        return len(words & _BANNED) / max(len(words), 1)
    TOX_SRC = "proxy-lista"

samples = [
    ("El transformador reduce el voltaje nominal.",        "técnico"),
    ("La regularización L2 penaliza los pesos grandes.",   "técnico"),
    ("Ese modelo es completamente inutil y basura.",        "ofensivo"),
    ("El sistema de potencia es estupido si no converge.", "mixto"),
]

print(f"Fuente: {TOX_SRC}")
print(f"{'Texto':50s}  {'Tipo':8s}  {'Score':>6}")
print("-" * 70)
for txt, typ in samples:
    sc = toxicity_score(txt)
    sc_val = sc if isinstance(sc, float) else float(sc)
    print(f"{txt[:50]:50s}  {typ:8s}  {sc_val:6.3f}")

print()
print("Limitaciones del proxy:")
print("  - No captura sarcasmo ('increíblemente brillante' como ironía)")
print("  - Sensible al dominio (palabras técnicas pueden coincidir con lista)")
print("  - Para producción: usar detoxify o Perspective API con calibración")

## 9. Sesgo — WEAT (Word Embedding Association Test)

### Definición formal (Caliskan et al., 2017)

Sean:
- $X, Y$: conjuntos de palabras **objetivo** (e.g., profesiones)
- $A, B$: conjuntos de palabras **atributo** (e.g., masculino/femenino)

La **asociación diferencial** de una palabra $w$ es:

$$s(w, A, B) = \text{mean}_{a\in A}\cos(\mathbf{w}, \mathbf{a}) - \text{mean}_{b\in B}\cos(\mathbf{w}, \mathbf{b})$$

El **estadístico WEAT** (effect size, como d de Cohen):

$$\text{effect size} = \frac{\text{mean}_{x\in X} s(x,A,B) - \text{mean}_{y\in Y} s(y,A,B)}{\text{std}_{w\in X\cup Y} s(w,A,B)}$$

Rango típico: $[-2, 2]$. Valor $> 0.5$ se considera sesgo notable.

El **p-valor** se estima por permutación: barajando $X$ e $Y$ repetidamente.

In [ ]:
# WEAT con embeddings de 8 dimensiones y sesgo plantado
rng_weat = np.random.default_rng(7)

def make_emb_weat(word, gender_bias, domain_bias, d=8):
    """Embedding con sesgo de género en dim 0 y dominio en dim 1."""
    base = rng_weat.standard_normal(d)
    base[0] = gender_bias * 3.0   # eje de género
    base[1] = domain_bias * 2.0   # eje de dominio técnico
    return base / (np.linalg.norm(base) + 1e-9)

# Objetivos: profesiones
TARGET_X = {  # sesgo "masculino" plantado
    "ingeniero":    make_emb_weat("ingeniero",    0.9,  0.8),
    "programador":  make_emb_weat("programador",  0.7,  0.7),
    "fisico":       make_emb_weat("fisico",        0.6,  0.9),
    "matematico":   make_emb_weat("matematico",   0.5,  0.9),
}
TARGET_Y = {  # sesgo "femenino" plantado
    "enfermera":    make_emb_weat("enfermera",   -0.9,  0.3),
    "maestra":      make_emb_weat("maestra",     -0.7,  0.4),
    "recepcionista":make_emb_weat("recepcionista",-0.6, 0.1),
    "costurera":    make_emb_weat("costurera",   -0.5, -0.1),
}
# Atributos
ATTR_A = {  # masculino
    "el":     make_emb_weat("el",    1.0, 0.0),
    "hombre": make_emb_weat("hombre",1.0, 0.0),
    "padre":  make_emb_weat("padre", 0.9, 0.0),
}
ATTR_B = {  # femenino
    "ella":  make_emb_weat("ella",  -1.0, 0.0),
    "mujer": make_emb_weat("mujer", -1.0, 0.0),
    "madre": make_emb_weat("madre", -0.9, 0.0),
}

def cos_weat(v1, v2):
    return float(np.dot(v1, v2))  # ya normalizados

def assoc(word_emb, A_dict, B_dict):
    ma = np.mean([cos_weat(word_emb, a) for a in A_dict.values()])
    mb = np.mean([cos_weat(word_emb, b) for b in B_dict.values()])
    return ma - mb

# Calcular asociación para todos los objetivos
all_targets = {**TARGET_X, **TARGET_Y}
assoc_scores = {w: assoc(e, ATTR_A, ATTR_B) for w, e in all_targets.items()}

# Effect size
s_X = np.array([assoc(e, ATTR_A, ATTR_B) for e in TARGET_X.values()])
s_Y = np.array([assoc(e, ATTR_A, ATTR_B) for e in TARGET_Y.values()])
s_all = np.concatenate([s_X, s_Y])
effect_size = (s_X.mean() - s_Y.mean()) / (s_all.std() + 1e-9)

print(f"Asociación por palabra:")
for w, sc in sorted(assoc_scores.items(), key=lambda x: x[1], reverse=True):
    bar = "#" * int(abs(sc) * 20)
    direction = "← femenino" if sc < 0 else "masculino →"
    print(f"  {w:15s}: {sc:+.3f}  {bar:20s} {direction}")
print(f"\nEffect size WEAT = {effect_size:.3f}  (|d| > 0.5 = sesgo notable)")

# p-valor por permutación
n_perm = 5000
perm_stats = []
all_assoc = list(s_X) + list(s_Y)
nX = len(s_X)
rng_perm = np.random.default_rng(99)
for _ in range(n_perm):
    shuffled = rng_perm.permutation(all_assoc)
    perm_stats.append(shuffled[:nX].mean() - shuffled[nX:].mean())
p_val = np.mean(np.array(perm_stats) >= (s_X.mean() - s_Y.mean()))
print(f"p-valor (permutación, n={n_perm}): {p_val:.4f}")

In [ ]:
# Barras divergentes por profesión
words_sorted = sorted(assoc_scores.items(), key=lambda x: x[1])
ws = [w for w, _ in words_sorted]
vs = [v for _, v in words_sorted]
colors_weat = ["#c0392b" if v < 0 else "#2980b9" for v in vs]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(ws, vs, color=colors_weat)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Asociación diferencial  (← femenino · masculino →)")
ax.set_title(f"WEAT: asociación profesión–género (embeddings 8-dim)\n"
             f"Effect size d = {effect_size:.2f}  |  p = {p_val:.4f}")
for bar, val in zip(bars, vs):
    ax.text(val + (0.01 if val >= 0 else -0.01),
            bar.get_y() + bar.get_height()/2,
            f"{val:+.2f}", va="center", ha="left" if val >= 0 else "right", fontsize=9)
plt.tight_layout(); plt.show()

print()
print("Advertencia metodológica:")
print("  1. WEAT cuantifica asociación estadística, NO causa ni intención.")
print("  2. El resultado depende fuertemente de los conjuntos A, B, X, Y elegidos.")
print("  3. En embeddings reales (word2vec, GloVe, BERT) se han reportado sesgos")
print("     similares que reflejan corpus de entrenamiento con sesgos históricos.")

## 10. Robustez

Un modelo robusto mantiene su rendimiento ante perturbaciones superficiales.
El siguiente harness aplica varios tipos de perturbación sobre un texto de referencia
y mide la degradación de BLEU y ROUGE-L frente al texto original.

### Tipos de perturbación
| Tipo | Descripción |
|------|-------------|
| `typos` | Reemplaza caracteres aleatoriamente |
| `uppercase` | Convierte a mayúsculas |
| `extra_spaces` | Inserta espacios extra entre palabras |
| `delete_words` | Elimina el 20% de palabras aleatoriamente |
| `swap_words` | Intercambia pares de palabras adyacentes |
| `punct_noise` | Inserta puntuación aleatoria |
| `homoglyphs` | Reemplaza letras por homóglifos similares |

Un LM robusto debería tener PPL similar bajo todas estas condiciones.

In [ ]:
rng_rob = random.Random(SEED)

def perturb_typos(text, p=0.12):
    chars = list(text)
    for i in range(len(chars)):
        if rng_rob.random() < p and chars[i].isalpha():
            chars[i] = rng_rob.choice("aeioubcdfgmnprstv")
    return "".join(chars)

def perturb_uppercase(text):
    return text.upper()

def perturb_extra_spaces(text):
    return "  ".join(text.split())

def perturb_delete_words(text, p=0.2):
    words = text.split()
    kept = [w for w in words if rng_rob.random() > p]
    return " ".join(kept) if kept else text

def perturb_swap_words(text):
    words = text.split()
    for i in range(0, len(words)-1, 2):
        words[i], words[i+1] = words[i+1], words[i]
    return " ".join(words)

def perturb_punct_noise(text):
    puncts = [",", ".", ";", "!"]
    words = text.split()
    out = []
    for w in words:
        out.append(w)
        if rng_rob.random() < 0.15:
            out.append(rng_rob.choice(puncts))
    return " ".join(out)

def perturb_homoglyphs(text):
    table = str.maketrans("aeiouAEIOU", "4310043100")
    return text.translate(table)

PERTURBATIONS = {
    "original":    lambda t: t,
    "typos":       perturb_typos,
    "uppercase":   perturb_uppercase,
    "extra_spaces":perturb_extra_spaces,
    "delete_words":perturb_delete_words,
    "swap_words":  perturb_swap_words,
    "punct_noise": perturb_punct_noise,
    "homoglyphs":  perturb_homoglyphs,
}

base = ("el transformador de potencia reduce el voltaje de transmision "
        "al nivel de distribucion para suministrar energia a los usuarios")

results_rob = {}
for name, fn in PERTURBATIONS.items():
    perturbed = fn(base)
    b   = bleu_full(perturbed, base, verbose=False)
    rl  = rouge_l(perturbed, base)["F"]
    results_rob[name] = {"BLEU": b, "ROUGE-L": rl, "text": perturbed[:60]}

print(f"{'Perturbación':14s}  {'BLEU':>6}  {'ROUGE-L':>8}  Texto (60 chars)")
print("-" * 90)
for k, v in results_rob.items():
    print(f"{k:14s}  {v['BLEU']:6.3f}  {v['ROUGE-L']:8.3f}  {v['text']}")

In [ ]:
# Gráfico de degradación por tipo de perturbación
pert_names = list(results_rob.keys())
bleu_vals   = [results_rob[k]["BLEU"]    for k in pert_names]
rouge_vals  = [results_rob[k]["ROUGE-L"] for k in pert_names]

x_rob = np.arange(len(pert_names))
w_rob = 0.35

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x_rob - w_rob/2, bleu_vals,  w_rob, label="BLEU",    color="#1a6fa8")
ax.bar(x_rob + w_rob/2, rouge_vals, w_rob, label="ROUGE-L", color="#3aaa55")
ax.axhline(1.0, color="gray", ls="--", lw=0.8, label="máximo teórico")
ax.set_xticks(x_rob); ax.set_xticklabels(pert_names, rotation=20, ha="right")
ax.set_ylabel("Score vs original"); ax.set_ylim(0, 1.1)
ax.set_title("Degradación de métricas por tipo de perturbación\n"
             "(original = 1.0; menor = más sensible a esa perturbación)")
ax.legend(); plt.tight_layout(); plt.show()

print("Observación: 'delete_words' y 'swap_words' degradan ROUGE-L más que BLEU.")
print("'homoglyphs' destruye BLEU pero mantiene ROUGE-L parcialmente.")

## 11. Benchmarks estándar

| Benchmark | Tarea | Formato | # preguntas |
|-----------|-------|---------|-------------|
| **MMLU** | Conocimiento 57 materias | 4-choice | 14 000 |
| **HellaSwag** | Sentido común (completar) | 4-choice | 10 003 |
| **ARC-Easy/Challenge** | Razonamiento científico | 4-choice | 7 787 |
| **GSM8K** | Matemáticas paso a paso | respuesta libre | 8 500 |
| **HumanEval** | Generación de código | tests unitarios | 164 |
| **TruthfulQA** | Veracidad | generación+MC | 817 |
| **BIG-Bench** | 204 tareas diversas | variado | ~150 K |

### lm-evaluation-harness (EleutherAI)

```bash
lm_eval --model hf --model_args pretrained=gpt2 \
        --tasks mmlu,arc_easy,hellaswag \
        --device cuda --batch_size 8
```

### Problemas de los benchmarks estándar
1. **Contaminación**: MMLU y otros se han filtrado a pretraining data de modelos grandes.
2. **Sensibilidad al prompt**: formatear la pregunta con "Answer:" vs "Respuesta:" mueve ±5 pp.
3. **Memorización vs razonamiento**: un modelo puede memorizar benchmarks sin razonar.
4. **Sesgo de dominio**: MMLU cubre materias de cultura anglosajona, poca representación técnica latina.

> Por eso construimos nuestro propio benchmark en la siguiente sección.

## 12. Benchmark propio del lab

### Diseño
- **~50 preguntas** multiple-choice (4 opciones), balanceadas entre los 21 dominios.
- Cada pregunta tiene una respuesta correcta claramente verificable.
- Las opciones incorrectas son **distractores plausibles** del mismo dominio.
- Evaluación por **log-verosimilitud**: se elige la opción con mayor $\log p(\text{opción} \mid \text{pregunta})$.
  Así el resultado es determinista e independiente del formato de salida.
- Reportamos accuracy por dominio con **intervalos de Wilson** al 95%.

### Intervalo de Wilson

$$\hat{p} = \frac{c}{n}, \quad \tilde{p} = \frac{\hat{p} + z^2/(2n)}{1 + z^2/n}, \quad
\text{margen} = \frac{z\sqrt{\hat{p}(1-\hat{p})/n + z^2/(4n^2)}}{1+z^2/n}$$

con $z = 1.96$ para 95%. Más robusto que el intervalo normal cuando $n$ es pequeño.

In [ ]:
# Benchmark propio: ~50 preguntas de los 21 dominios
BENCHMARK = [
    # machine_learning (5)
    {"q":"¿Qué objetivo minimiza el descenso de gradiente estocástico?",
     "opts":["La función de pérdida","La VRAM","El número de capas","El learning rate"],
     "a":0,"dom":"machine_learning"},
    {"q":"¿Cuál es la diferencia principal entre sesgo y varianza?",
     "opts":["Sesgo=error sistemático, varianza=sensibilidad a datos",
             "Sesgo=varianza, varianza=sesgo",
             "Sesgo mide overfitting, varianza mide underfitting",
             "No hay diferencia"],
     "a":0,"dom":"machine_learning"},
    {"q":"¿Qué hace el dropout durante el entrenamiento?",
     "opts":["Desactiva neuronas aleatoriamente","Aumenta el learning rate",
             "Reduce el batch size","Elimina capas"],
     "a":0,"dom":"machine_learning"},
    {"q":"¿Cuándo se prefiere Random Forest sobre una regresión lineal?",
     "opts":["Relaciones no lineales entre variables",
             "Siempre que el dataset sea pequeño",
             "Cuando hay pocas features",
             "Cuando la variable objetivo es categórica"],
     "a":0,"dom":"machine_learning"},
    {"q":"¿Qué mide la curva ROC-AUC?",
     "opts":["Capacidad de discriminar clases positivas de negativas",
             "El error cuadrático medio",
             "La complejidad del modelo",
             "El tiempo de inferencia"],
     "a":0,"dom":"machine_learning"},

    # deep_learning (4)
    {"q":"¿Qué calcula la retropropagación?",
     "opts":["Gradientes de la pérdida respecto a los pesos",
             "Los pesos óptimos directamente",
             "La función de activación",
             "El número de capas necesarias"],
     "a":0,"dom":"deep_learning"},
    {"q":"¿Qué problema resuelve la normalización por lotes (Batch Norm)?",
     "opts":["Inestabilidad del entrenamiento por cambio de distribución interna",
             "El overfitting",
             "La falta de datos",
             "La lentitud del hardware"],
     "a":0,"dom":"deep_learning"},
    {"q":"¿Cuál es la complejidad temporal de la self-attention?",
     "opts":["O(T²·d)","O(T·d)","O(d²)","O(T·log T)"],
     "a":0,"dom":"deep_learning"},
    {"q":"¿Para qué sirve la capa de embedding en un LLM?",
     "opts":["Mapear tokens a vectores densos",
             "Calcular la atención",
             "Normalizar los pesos",
             "Reducir el vocabulario"],
     "a":0,"dom":"deep_learning"},

    # sistemas_potencia (5)
    {"q":"¿Qué método se usa para resolver el flujo de carga en sistemas de potencia?",
     "opts":["Newton-Raphson","K-means","Transformada de Laplace","Regresión logística"],
     "a":0,"dom":"sistemas_potencia"},
    {"q":"¿Qué es la estabilidad transitoria en un sistema eléctrico?",
     "opts":["Capacidad de mantener sincronismo tras una perturbación severa",
             "La tensión nominal de la línea",
             "La impedancia de la red",
             "El factor de potencia"],
     "a":0,"dom":"sistemas_potencia"},
    {"q":"¿Cuál es la ventaja de alta tensión en transmisión?",
     "opts":["Menores pérdidas por efecto Joule",
             "Mayor velocidad de la corriente",
             "Menor reactancia inductiva",
             "Reducción del voltaje de cortocircuito"],
     "a":0,"dom":"sistemas_potencia"},
    {"q":"¿Qué ley se aplica en el análisis de nodos?",
     "opts":["Ley de Kirchhoff de corrientes (KCL)",
             "Ley de Faraday",
             "Ley de Lenz",
             "Ley de Ohm únicamente"],
     "a":0,"dom":"sistemas_potencia"},
    {"q":"¿Qué representa el ángulo de carga en una máquina síncrona?",
     "opts":["El ángulo entre el fasor de tensión y la FEM interna",
             "La frecuencia del rotor",
             "La resistencia del estátor",
             "El número de polos"],
     "a":0,"dom":"sistemas_potencia"},

    # ingenieria_electrica (3)
    {"q":"¿Qué expresa la ley de Ohm?",
     "opts":["V = I·R","P = V²/R","Q = C·V","F = q·E"],
     "a":0,"dom":"ingenieria_electrica"},
    {"q":"¿En qué condición resuena un circuito RLC serie?",
     "opts":["Cuando XL = XC","Cuando R = 0","Cuando V = I","Cuando f = 0"],
     "a":0,"dom":"ingenieria_electrica"},
    {"q":"¿Qué convierte una máquina síncrona como generador?",
     "opts":["Energía mecánica en eléctrica alterna",
             "CC en CA","CA en CC","Calor en electricidad"],
     "a":0,"dom":"ingenieria_electrica"},

    # matematicas (4)
    {"q":"¿Qué factoriza la SVD?",
     "opts":["A = UΣVᵀ","A = LU","A = QR","A = PDP⁻¹"],
     "a":0,"dom":"matematicas"},
    {"q":"¿Qué establece el teorema de Bayes?",
     "opts":["P(A|B) = P(B|A)·P(A)/P(B)",
             "P(A|B) = P(A)·P(B)",
             "P(A∪B) = P(A)+P(B)-P(A∩B)",
             "P(A) = 1-P(Aᶜ)"],
     "a":0,"dom":"matematicas"},
    {"q":"¿Qué descompone la transformada de Fourier?",
     "opts":["Señales en componentes de frecuencia",
             "Matrices en valores propios",
             "Vectores en bases ortogonales",
             "Funciones en series de Taylor"],
     "a":0,"dom":"matematicas"},
    {"q":"¿Cuándo una función es convexa?",
     "opts":["Cuando la segunda derivada es no negativa",
             "Cuando es creciente",
             "Cuando es acotada",
             "Cuando es diferenciable"],
     "a":0,"dom":"matematicas"},

    # estadistica (3)
    {"q":"¿Qué dice el teorema central del límite?",
     "opts":["La media muestral se distribuye normalmente para n grande",
             "Toda distribución es normal",
             "La varianza siempre tiende a 0",
             "Las muestras son independientes"],
     "a":0,"dom":"estadistica"},
    {"q":"¿Qué es el p-valor?",
     "opts":["Probabilidad de observar datos tan extremos si H0 es cierta",
             "La probabilidad de que H1 sea verdadera",
             "El nivel de significancia elegido",
             "La potencia del test"],
     "a":0,"dom":"estadistica"},
    {"q":"¿Qué mide la correlación de Pearson?",
     "opts":["Relación lineal entre dos variables",
             "Relación no lineal",
             "Causalidad entre variables",
             "La varianza conjunta"],
     "a":0,"dom":"estadistica"},

    # optimizacion (3)
    {"q":"¿Qué combina el optimizador Adam?",
     "opts":["Momentum y adaptación de lr por parámetro",
             "Solo momentum","Solo RMSProp","Gradiente y Hessiano"],
     "a":0,"dom":"optimizacion"},
    {"q":"¿Qué garantiza un problema de optimización convexo?",
     "opts":["Que todo mínimo local es global",
             "Que siempre hay solución única",
             "Que el gradiente es cero en todas partes",
             "Que la función es lineal"],
     "a":0,"dom":"optimizacion"},
    {"q":"¿Qué es el learning rate scheduling?",
     "opts":["Reducir el lr durante el entrenamiento según una política",
             "Aumentar el batch size",
             "Cambiar la arquitectura del modelo",
             "Seleccionar el optimizador"],
     "a":0,"dom":"optimizacion"},

    # python (3)
    {"q":"¿Qué es un generador en Python?",
     "opts":["Función que produce valores de forma lazy con yield",
             "Función que retorna una lista",
             "Clase con __iter__",
             "Módulo de generación aleatoria"],
     "a":0,"dom":"python"},
    {"q":"¿Qué hace un decorador en Python?",
     "opts":["Modifica el comportamiento de una función",
             "Declara variables globales",
             "Importa módulos dinámicamente",
             "Compila el código"],
     "a":0,"dom":"python"},
    {"q":"¿Qué ventaja tiene asyncio sobre threading en I/O bound?",
     "opts":["Menor overhead de contexto y mejor uso del event loop",
             "Usa múltiples CPUs",
             "Es más rápido en cómputo",
             "Simplifica el código siempre"],
     "a":0,"dom":"python"},

    # postgresql (3)
    {"q":"¿Qué estructura usa el índice por defecto en PostgreSQL?",
     "opts":["B-tree","Hash","BRIN","GiST"],
     "a":0,"dom":"postgresql"},
    {"q":"¿Para qué sirve VACUUM en PostgreSQL?",
     "opts":["Recuperar espacio de tuplas muertas",
             "Reconstruir índices",
             "Optimizar consultas",
             "Crear backups"],
     "a":0,"dom":"postgresql"},
    {"q":"¿Qué garantizan las transacciones ACID?",
     "opts":["Atomicidad, Consistencia, Aislamiento, Durabilidad",
             "Alta disponibilidad",
             "Escalabilidad horizontal",
             "Baja latencia"],
     "a":0,"dom":"postgresql"},

    # aws (3)
    {"q":"¿Cómo se factura AWS Lambda?",
     "opts":["Por milisegundos de ejecución y memoria usada",
             "Por servidor/mes",
             "Por número de usuarios",
             "Es gratuito siempre"],
     "a":0,"dom":"aws"},
    {"q":"¿Qué durabilidad ofrece Amazon S3?",
     "opts":["99.999999999% (11 nueves)",
             "99.9%","100%","99.99%"],
     "a":0,"dom":"aws"},
    {"q":"¿Qué hace CloudFormation?",
     "opts":["Define infraestructura como código mediante plantillas",
             "Monitorea costos en tiempo real",
             "Escala automáticamente las instancias",
             "Gestiona usuarios IAM"],
     "a":0,"dom":"aws"},

    # arquitectura_software (3)
    {"q":"¿Qué separa el patrón CQRS?",
     "opts":["Operaciones de lectura y escritura",
             "Frontend y backend",
             "Microservicios y monolito",
             "Base de datos y caché"],
     "a":0,"dom":"arquitectura_software"},
    {"q":"¿Qué previene el patrón Circuit Breaker?",
     "opts":["Cascadas de fallos en sistemas distribuidos",
             "Inyección SQL",
             "Fugas de memoria",
             "Deadlocks"],
     "a":0,"dom":"arquitectura_software"},
    {"q":"¿Qué almacena event sourcing?",
     "opts":["El estado como secuencia de eventos inmutables",
             "Snapshots periódicos del estado",
             "Solo el estado actual",
             "Logs de errores"],
     "a":0,"dom":"arquitectura_software"},

    # devops (3)
    {"q":"¿Qué garantiza la integración continua?",
     "opts":["Validación automática de cada commit",
             "Despliegue automático a producción",
             "Reducción del número de bugs",
             "Escalado de servidores"],
     "a":0,"dom":"devops"},
    {"q":"¿Qué orquesta Kubernetes?",
     "opts":["Contenedores con escalado y balanceo de carga",
             "Máquinas virtuales",
             "Bases de datos NoSQL",
             "Funciones serverless"],
     "a":0,"dom":"devops"},
    {"q":"¿Qué portabilidad ofrece Docker?",
     "opts":["El mismo entorno en dev, staging y producción",
             "Acceso multi-cloud automático",
             "Reducción de latencia de red",
             "Cifrado de datos"],
     "a":0,"dom":"devops"},

    # data_engineering (3)
    {"q":"¿Qué procesan los pipelines ETL?",
     "opts":["Extraen, transforman y cargan datos",
             "Entrenan modelos ML",
             "Gestionan usuarios",
             "Monitorean infraestructura"],
     "a":0,"dom":"data_engineering"},
    {"q":"¿Qué procesa Apache Spark en paralelo?",
     "opts":["Grandes volúmenes de datos sobre un clúster",
             "Consultas SQL en memoria",
             "Solo datos estructurados",
             "Modelos de deep learning"],
     "a":0,"dom":"data_engineering"},
    {"q":"¿Qué organiza el modelado dimensional?",
     "opts":["Datos en tablas de hechos y dimensiones",
             "Datos en grafos",
             "Documentos JSON",
             "Series temporales"],
     "a":0,"dom":"data_engineering"},

    # sistemas_distribuidos (3)
    {"q":"¿Qué establece el teorema CAP?",
     "opts":["No se puede garantizar C, A y P simultáneamente",
             "Todo sistema distribuido es eventualmente consistente",
             "La disponibilidad es siempre mayor que la consistencia",
             "Las particiones nunca ocurren"],
     "a":0,"dom":"sistemas_distribuidos"},
    {"q":"¿Qué hace el algoritmo Raft?",
     "opts":["Consenso distribuido con elección de líder",
             "Balanceo de carga",
             "Cifrado de mensajes",
             "Compresión de datos"],
     "a":0,"dom":"sistemas_distribuidos"},
    {"q":"¿Qué permite la consistencia eventual?",
     "opts":["Mayor disponibilidad a cambio de divergencia temporal",
             "Lecturas siempre actualizadas",
             "Transacciones ACID distribuidas",
             "Latencia cero"],
     "a":0,"dom":"sistemas_distribuidos"},
]

print(f"Total de preguntas: {len(BENCHMARK)}")
dom_counts = Counter(q["dom"] for q in BENCHMARK)
for dom, cnt in sorted(dom_counts.items()):
    print(f"  {dom:30s}: {cnt}")

In [ ]:
# score_fn proxy (en producción: reemplazar por logprob del modelo)
rng_bench = random.Random(SEED + 1)

def score_option(question: str, option: str) -> float:
    """
    Proxy determinista del log-verosimilitud de una opción dado el contexto.
    La opción correcta tiene más solapamiento léxico con la pregunta + ruido pequeño.
    En producción se usa: model(question + option, labels=option_ids).loss
    """
    qw = set(question.lower().split())
    ow = set(option.lower().split())
    overlap = len(qw & ow) / max(len(ow), 1)
    # sesgo leve hacia opciones más largas (simula preferencia del modelo)
    length_bonus = len(option.split()) * 0.05
    noise = rng_bench.gauss(0, 0.3)
    return overlap + length_bonus + noise


def wilson_ci(correct, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = correct / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    margin = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / denom
    return (max(0.0, center - margin), min(1.0, center + margin))


# Evaluar con CONFIG["bench_reps"] repeticiones (barajando distractores)
bench_results = defaultdict(lambda: [0, 0])  # dom -> [correct, total]
total_correct = 0

for rep in range(CONFIG["bench_reps"]):
    for item in BENCHMARK:
        opts = list(item["opts"])
        correct_opt = opts[item["a"]]
        rng_bench.shuffle(opts)
        new_a = opts.index(correct_opt)
        scores = [score_option(item["q"], o) for o in opts]
        pred = int(np.argmax(scores))
        ok = (pred == new_a)
        total_correct += ok
        bench_results[item["dom"]][0] += ok
        bench_results[item["dom"]][1] += 1

total_n = len(BENCHMARK) * CONFIG["bench_reps"]
global_acc = total_correct / total_n
lo_g, hi_g = wilson_ci(total_correct, total_n)
print(f"Accuracy global: {global_acc:.2%}  Wilson 95% CI: [{lo_g:.2%}, {hi_g:.2%}]  (n={total_n})")
print()
print(f"{'Dominio':30s}  {'Acc':>6}  {'CI 95%':>14}  {'n':>4}")
print("-" * 60)
for dom in sorted(bench_results):
    c, n = bench_results[dom]
    acc  = c/n if n>0 else 0
    lo, hi = wilson_ci(c, n)
    print(f"{dom:30s}  {acc:6.2%}  [{lo:.2%}, {hi:.2%}]  {n:4d}")

In [ ]:
# Barras con intervalos de Wilson
doms_bench = sorted(bench_results.keys())
accs_b = [bench_results[d][0]/bench_results[d][1] for d in doms_bench]
los_b  = [wilson_ci(bench_results[d][0], bench_results[d][1])[0] for d in doms_bench]
his_b  = [wilson_ci(bench_results[d][0], bench_results[d][1])[1] for d in doms_bench]
errs_lo = [a - l for a, l in zip(accs_b, los_b)]
errs_hi = [h - a for a, h in zip(accs_b, his_b)]

# Color por peso de dominio
bar_colors = [palette.get(DOMAIN_WEIGHTS.get(d, 1.0), "#888") for d in doms_bench]

fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(doms_bench, accs_b, color=bar_colors, alpha=0.85)
ax.errorbar(doms_bench, accs_b,
            yerr=[errs_lo, errs_hi],
            fmt="none", color="black", capsize=4, linewidth=1.5)
ax.axhline(0.25, color="red", ls="--", lw=1, label="azar (4 opciones)")
ax.axhline(global_acc, color="navy", ls=":", lw=1.5,
           label=f"media global {global_acc:.1%}")
ax.set_xticks(range(len(doms_bench))); ax.set_xticklabels(doms_bench, rotation=45, ha="right")
ax.set_ylabel("Accuracy"); ax.set_ylim(0, 1.05)
ax.set_title("Benchmark propio — Accuracy por dominio con intervalo de Wilson 95%\n"
             "Azul=peso 1.5 · Verde=peso 1.2 · Naranja=peso 1.0")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Radar chart Plotly por dominio (requiere plotly)
if PLOTLY:
    cats = doms_bench + [doms_bench[0]]  # cerrar el polígono
    vals = accs_b + [accs_b[0]]

    fig_radar = go.Figure()
    fig_radar.add_trace(go.Scatterpolar(
        r=vals, theta=cats, fill="toself",
        name="score_fn proxy",
        line_color="#1a6fa8", fillcolor="rgba(26,111,168,0.2)"
    ))
    # Línea de azar
    chance = [0.25] * len(cats)
    fig_radar.add_trace(go.Scatterpolar(
        r=chance, theta=cats, mode="lines",
        name="azar (0.25)", line=dict(color="red", dash="dash")
    ))
    fig_radar.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        title="Benchmark propio — Radar por dominio",
        showlegend=True, height=550
    )
    fig_radar.show()
else:
    # Fallback matplotlib
    N = len(doms_bench)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]
    vals_radar = accs_b + accs_b[:1]
    fig_r, ax_r = plt.subplots(subplot_kw={"projection":"polar"}, figsize=(7,7))
    ax_r.plot(angles, vals_radar, "o-", lw=2)
    ax_r.fill(angles, vals_radar, alpha=0.25)
    ax_r.set_xticks(angles[:-1]); ax_r.set_xticklabels(doms_bench, fontsize=7)
    ax_r.set_title("Radar por dominio (matplotlib fallback)")
    plt.tight_layout(); plt.show()

In [ ]:
# Heatmap modelo × dominio (simulamos 3 modelos para ilustrar)
model_names = ["score_fn-proxy", "random-baseline", "perfect-model"]

def simulate_model_acc(model_idx, domain):
    rng_m = np.random.default_rng(model_idx * 100 + abs(hash(domain)) % 1000)
    base  = [global_acc, 0.25, 1.0][model_idx]
    noise = rng_m.normal(0, [0.05, 0.03, 0.0][model_idx])
    return float(np.clip(base + noise, 0, 1))

heat_matrix = np.array([
    [simulate_model_acc(mi, dom) for dom in doms_bench]
    for mi in range(len(model_names))
])

fig, ax = plt.subplots(figsize=(14, 3.5))
sns.heatmap(heat_matrix, annot=True, fmt=".2f", cmap="YlGn",
            xticklabels=doms_bench, yticklabels=model_names,
            vmin=0, vmax=1, ax=ax, cbar_kws={"label": "accuracy"})
ax.set_title("Heatmap Modelo × Dominio\n(modelos simulados; en producción: reemplazar por logprob real)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
plt.tight_layout(); plt.show()

## 13. Comparación entre checkpoints / modelos

Durante el entrenamiento se guardan checkpoints periódicos. Para cada uno se mide:
- **PPL en val set**: señal directa del loss de entrenamiento.
- **Accuracy en el benchmark propio**: señal de generalización.

La curva PPL ↓ + accuracy ↑ valida que el modelo aprende y no solo memoriza.
Si PPL sigue bajando pero accuracy se estanca, puede haber **memorización sin generalización**.

Una comparación justa entre modelos requiere:
1. Mismo número de tokens vistos (no mismo número de steps si el batch varía).
2. Mismo benchmark con el mismo prompt format.
3. Mismos seeds para los distractores.

In [ ]:
# Curvas simuladas (en producción se cargan desde logs de WandB/tensorboard)
steps = np.arange(0, 5001, 500)
rng_curves = np.random.default_rng(42)

# modelo SMALL
ppl_small  = 80 * np.exp(-steps/1200) + 5 + rng_curves.normal(0, 0.3, len(steps))
acc_small  = np.clip(0.25 + 0.55*(1 - np.exp(-steps/1500)), 0, 1)

# modelo LARGE (converge más lento pero a mejor punto)
ppl_large  = 80 * np.exp(-steps/2000) + 3 + rng_curves.normal(0, 0.2, len(steps))
acc_large  = np.clip(0.25 + 0.68*(1 - np.exp(-steps/2000)), 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(steps, ppl_small, "o-", label="SMALL (~4M)", color="#1a6fa8")
axes[0].plot(steps, ppl_large, "s-", label="LARGE (~85M)", color="#c47a1e")
axes[0].set_xlabel("step"); axes[0].set_ylabel("perplexity (val)")
axes[0].set_title("PPL durante el entrenamiento")
axes[0].legend()

axes[1].plot(steps, acc_small, "o-", label="SMALL", color="#1a6fa8")
axes[1].plot(steps, acc_large, "s-", label="LARGE", color="#c47a1e")
axes[1].axhline(0.25, color="red", ls="--", lw=0.8, label="azar")
axes[1].set_xlabel("step"); axes[1].set_ylabel("accuracy benchmark")
axes[1].set_title("Accuracy benchmark durante el entrenamiento")
axes[1].legend()

plt.tight_layout(); plt.show()
print("Observación: LARGE llega a menor PPL y mayor accuracy, pero requiere más steps.")

## 14. Dashboard final de evaluación

La figura 2×3 siguiente reúne las métricas más importantes en un único vistazo:

| Posición | Métrica |
|----------|---------|
| (0,0) | PPL por dominio (barras) |
| (0,1) | Accuracy benchmark por dominio (barras + Wilson) |
| (0,2) | Adversariales: BLEU / ROUGE-L / BERTScore |
| (1,0) | Robustez por tipo de perturbación |
| (1,1) | WEAT: asociación profesión–género |
| (1,2) | Curvas PPL y accuracy vs steps (SMALL) |

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# (0,0) PPL por dominio
axes[0,0].barh(doms_sorted, ppl_sorted,
               color=[palette[DOMAIN_WEIGHTS.get(d,1.0)] for d in doms_sorted])
axes[0,0].set_title("PPL por dominio"); axes[0,0].tick_params(axis="y", labelsize=7)

# (0,1) Accuracy benchmark + Wilson CI
axes[0,1].bar(range(len(doms_bench)), accs_b,
              color=[palette.get(DOMAIN_WEIGHTS.get(d,1.0),"#888") for d in doms_bench],
              alpha=0.85)
axes[0,1].errorbar(range(len(doms_bench)), accs_b,
                   yerr=[errs_lo, errs_hi], fmt="none", color="black", capsize=3)
axes[0,1].axhline(0.25, color="red", ls="--", lw=0.8)
axes[0,1].set_xticks(range(len(doms_bench)))
axes[0,1].set_xticklabels(doms_bench, rotation=90, fontsize=6)
axes[0,1].set_title("Accuracy benchmark (Wilson CI 95%)")
axes[0,1].set_ylim(0, 1.05)

# (0,2) Adversariales: 3 métricas
x_adv = np.arange(len(results_adv))
w_adv = 0.28
axes[0,2].bar(x_adv-w_adv, [r["BLEU"]     for r in results_adv], w_adv, label="BLEU",    color="#1a6fa8")
axes[0,2].bar(x_adv,       [r["ROUGE-L"]  for r in results_adv], w_adv, label="ROUGE-L", color="#3aaa55")
axes[0,2].bar(x_adv+w_adv, [r["BERTScore"]for r in results_adv], w_adv, label="BERTScore",color="#c47a1e")
axes[0,2].set_xticks(x_adv)
axes[0,2].set_xticklabels([r["tipo"][:6] for r in results_adv], rotation=30, fontsize=7)
axes[0,2].set_title("Pares adversariales"); axes[0,2].set_ylim(0,1.05); axes[0,2].legend(fontsize=7)

# (1,0) Robustez
pert_n = list(results_rob.keys())
bleu_r  = [results_rob[k]["BLEU"]    for k in pert_n]
rouge_r = [results_rob[k]["ROUGE-L"] for k in pert_n]
x_r = np.arange(len(pert_n)); w_r = 0.35
axes[1,0].bar(x_r-w_r/2, bleu_r,  w_r, label="BLEU",    color="#1a6fa8")
axes[1,0].bar(x_r+w_r/2, rouge_r, w_r, label="ROUGE-L", color="#3aaa55")
axes[1,0].set_xticks(x_r); axes[1,0].set_xticklabels(pert_n, rotation=30, fontsize=7, ha="right")
axes[1,0].set_title("Robustez vs perturbaciones"); axes[1,0].set_ylim(0,1.1); axes[1,0].legend(fontsize=7)

# (1,1) WEAT barras divergentes
ws_weat = list(assoc_scores.keys()); vs_weat = list(assoc_scores.values())
cols_d = ["#c0392b" if v<0 else "#2980b9" for v in vs_weat]
axes[1,1].barh(ws_weat, vs_weat, color=cols_d)
axes[1,1].axvline(0, color="black", lw=1)
axes[1,1].set_title(f"WEAT — effect size d={effect_size:.2f}")
axes[1,1].tick_params(axis="y", labelsize=8)

# (1,2) Curvas PPL y accuracy (SMALL)
ax2 = axes[1,2]
color_ppl = "#1a6fa8"
color_acc = "#3aaa55"
lns1 = ax2.plot(steps, ppl_small, "o-", color=color_ppl, label="PPL", lw=1.5)
ax2b = ax2.twinx()
lns2 = ax2b.plot(steps, acc_small, "s--", color=color_acc, label="Accuracy", lw=1.5)
ax2.set_ylabel("PPL", color=color_ppl); ax2b.set_ylabel("Accuracy", color=color_acc)
ax2.set_xlabel("step"); ax2.set_title("Curvas de entrenamiento (SMALL)")
lns = lns1 + lns2; labs = [l.get_label() for l in lns]
ax2.legend(lns, labs, fontsize=8, loc="upper right")

fig.suptitle("Dashboard de Evaluacion — LLM Lab (Fase 8)", fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

## 15. Resumen, ejercicios y referencias

### Resumen

| Métrica | Fortaleza | Debilidad |
|---------|-----------|-----------|
| **Perplexity** | Rápida, correlaciona con calidad | No comparable entre tokenizadores; no captura semántica |
| **BLEU** | Estándar en MT, determinista | Falla con paráfrasis y sinónimos |
| **ROUGE-L** | Captura estructura por LCS | Sensible al orden, no al significado |
| **BERTScore** | Captura significado | Costosa; depende del modelo de referencia |
| **WEAT** | Cuantifica sesgo de asociación | Sensible a los conjuntos elegidos |
| **Benchmark MC** | Determinista, escalable | Contaminación, sensibilidad al prompt |

El código modular está en [`src/llm_lab/evaluation/`](../src/llm_lab/evaluation/).

### Ejercicios propuestos

1. **(★)** Sustituye el `score_option` proxy por el logprob real de `sshleifer/tiny-gpt2`
   y re-evalúa el benchmark. ¿Cambia el ranking por dominio?

2. **(★★)** Implementa perplexity con sliding window real y grafica la curva PPL vs stride
   para strides de 1, 8, 16, 32, 64 tokens. ¿A partir de qué stride la PPL converge?

3. **(★★)** Añade 10 preguntas propias al benchmark en los dominios con menos cobertura
   (serverless, iac, mongodb). Verifica que los distractores sean plausibles pero incorrectos.

4. **(★★★)** Implementa detección de alucinaciones por NLI usando
   `cross-encoder/nli-deberta-v3-small` de HuggingFace. Mide la tasa de contradicción
   en 20 pares (pregunta, respuesta generada) del dominio eléctrico.

5. **(★★★)** Replica el test WEAT con embeddings reales de `sentence-transformers/all-MiniLM-L6-v2`.
   ¿El efecto observado coincide con lo reportado en la literatura para word2vec/GloVe?

### Referencias

- Papineni et al. (2002). *BLEU: a method for automatic evaluation of machine translation*.
  [ACL](https://aclanthology.org/P02-1040/)
- Lin (2004). *ROUGE: A Package for Automatic Evaluation of Summaries*.
  [ACL](https://aclanthology.org/W04-1013/)
- Zhang et al. (2019). *BERTScore: Evaluating Text Generation with BERT*.
  [arXiv:1904.09675](https://arxiv.org/abs/1904.09675)
- Ji et al. (2023). *Survey of Hallucination in Natural Language Generation*.
  [arXiv:2202.03629](https://arxiv.org/abs/2202.03629)
- Caliskan et al. (2017). *Semantics derived automatically from language corpora contain
  human-like biases*. [Science](https://www.science.org/doi/10.1126/science.aal4230)
- Gehman et al. (2020). *RealToxicityPrompts*. [arXiv:2009.11462](https://arxiv.org/abs/2009.11462)
- Hendrycks et al. (2021). *Measuring Massive Multitask Language Understanding (MMLU)*.
  [arXiv:2009.03300](https://arxiv.org/abs/2009.03300)
- Gao et al. (2021). *The Pile: An 800GB Dataset of Diverse Text for Language Modeling*.
  [arXiv:2101.00027](https://arxiv.org/abs/2101.00027)
- Wilson (1927). *Probable inference, the law of succession, and statistical inference*.
  JASA, 22(158), 209–212.